# 4 · WorldPop STAC: from discovery to RustGIS

**Question:** how can a search result become a map without confusing its footprint with raster values?

This lesson uses your [JupyterLite STAC Browser](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/) client and a frozen three-item WorldPop metadata snapshot retrieved 2026-10-14. These Items are **Degree of Urbanisation** products for Uganda, not population-count products. The snapshot is a reliability fallback; the optional live query may return different items later. No raster assets are bundled.

![STAC to RustGIS workflow](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-flow.svg)

![STAC to RustGIS workflow](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-flow.svg)

<video controls preload="metadata" width="760" src="https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-tour.mp4">A captioned five-step STAC to RustGIS explainer.</video>

[Open or download the captioned explainer video](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-tour.mp4).

**Video transcript:** Discover a public STAC collection; bound a search by area, date, and page size; inspect license and assets; export GeoJSON plus query provenance; open the result in RustGIS. Metadata footprints do not show raster pixel values.

## Inspect the query provenance

The snapshot preserves the source endpoint and exact bounded query. The Item geometries indicate coverage; asset links point to provider files and may have separate access terms.

In [ ]:
import json
from pathlib import Path
provenance = json.loads(Path('data/worldpop_provenance.json').read_text())
print(json.dumps(provenance, indent=2))

## Load the saved STAC Items

In [ ]:
snapshot = json.loads(Path('data/worldpop_uga_items.geojson').read_text())
items = snapshot['features']
print('Items:', len(items))
for item in items:
    print(item['id'], 'reference year:', item['properties'].get('year'), 'assets:', len(item.get('assets', {})))

## Separate metadata time from reference year

For WorldPop, a STAC `datetime` may be the release time. Compare it to `properties.year`; do not present the former as the population reference year.

In [ ]:
for item in items:
    p=item['properties']
    print(item['id'], '| year:', p.get('year'), '| datetime:', p.get('datetime'), '| project:', p.get('project'))

## Compare the years visually

The chart shows metadata labels, not population estimates.

In [ ]:
from IPython.display import display, HTML
from viz import bars, points
display(HTML(bars([i['id'] for i in items], [i['properties'].get('year',0) for i in items], 'Reference-year metadata', '', '#e6a468')))

## Map a footprint schematic

In [ ]:
display(HTML(points(items, 'STAC Item coverage (schematic)')))

## Inspect asset roles and links

Only print the first few names and media types. A STAC asset URL is not proof that an anonymous browser can download it.

In [ ]:
sample=items[0]
for key, asset in list(sample.get('assets', {}).items())[:8]:
    print(key, '|', asset.get('type'), '|', asset.get('roles'), '|', asset.get('href','')[:90])

## Optional live search through your STAC Browser client

The vendored client is copied from your STAC Browser repository under its MIT license. It uses browser fetch in Pyodide and a standard-library adapter locally. The query is bounded to a three-item page. Network/CORS errors leave the snapshot available.

In [ ]:
from stac_browser import STACBrowser, PRESETS
try:
    client = await STACBrowser(PRESETS['worldpop']).connect()
    live_items = await client.search(bbox=[29,-2,35,5], collection='UGA', limit=3)
    print('Live item IDs:', [item['id'] for item in live_items])
    print('Live provenance:', client.provenance().get('stac_root'))
except Exception as exc:
    live_items = []
    print('Live STAC search unavailable; continue with the saved snapshot:', type(exc).__name__, str(exc)[:180])

## Prepare a RustGIS-ready footprint export

GeoJSON features retain STAC properties. Strip asset fields from the teaching export so the map layer is small; keep provenance separately.

In [ ]:
footprints={'type':'FeatureCollection','features':[{'type':'Feature','id':i['id'],'geometry':i['geometry'],'properties':{'id':i['id'],'year':i['properties'].get('year'),'project':i['properties'].get('project')}} for i in items]}
Path('worldpop_uga_footprints.geojson').write_text(json.dumps(footprints, indent=2))
Path('worldpop_uga_query.json').write_text(json.dumps(provenance, indent=2))
print('Saved footprints and query provenance. Download both from the file browser.')

## Open in RustGIS

[Open the hosted STAC snapshot in RustGIS Web](https://web.rustgis.app/?blank&data=https://jltobias.github.io/JupyterLite-RustGIS/data/worldpop_uga_items.geojson). For the smaller export made above, download it and use RustGIS **Add Data**. Try symbolizing by `year`, then read the Item's provider and collection license before opening any raster asset.

**Interpretation check:** the map is a coverage footprint map. It does not display Degree of Urbanisation cell values or uncertainty.

**Citations:** [WorldPop STAC API](https://api.stac.worldpop.org), [your STAC Browser source](https://github.com/jltobias/JupyterLite-STAC-Browser), [STAC specification](https://stacspec.org/), [RustGIS web URL guide](https://github.com/opengeos/RustGIS/blob/main/docs/installation.md#web-app). See `data/worldpop_provenance.json` for the exact snapshot query.